# WaferVision — Full Training on Google Colab
## GPU: T4 (16 GB VRAM), RAM: 12 GB

This notebook runs the complete WaferVision pipeline:
1. Install dependencies
2. Upload/download WM-811K dataset
3. Preprocess all 172K labeled samples
4. Fine-tune ResNet50 (50 epochs)
5. Train Metric Learning (Triplet Loss, 50 epochs)
6. Evaluate all models
7. Generate benchmark comparison

In [ ]:
# Step 0: Clone repo and install
# Option A: Upload your wafer-vision folder as zip
# Option B: Clone from git (if you push to GitHub)

# Upload wafer-vision.zip from your PC:
from google.colab import files
uploaded = files.upload()  # Upload wafer-vision.zip

!unzip -q wafer-vision.zip -d /content/
%cd /content/wafer-vision
!pip install -e '.[dev]' -q

In [ ]:
# Step 0b: Upload LSWMD.pkl dataset
# Option A: Upload directly (2 GB — takes a while)
# Option B: Download from Kaggle

# === OPTION A: Direct upload ===
# from google.colab import files
# uploaded = files.upload()  # Upload LSWMD.pkl
# !mkdir -p data/raw && mv LSWMD.pkl data/raw/

# === OPTION B: From Kaggle (recommended) ===
# 1. Go to kaggle.com -> Account -> Create New API Token
# 2. Upload kaggle.json
!pip install kaggle -q
from google.colab import files
uploaded = files.upload()  # Upload kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d qingyi/wm811k-wafer-map -p data/raw/ --unzip
!ls -la data/raw/

In [ ]:
# Step 1: Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Step 2: Load and preprocess ALL 172K labeled samples
import sys, time
sys.path.insert(0, '.')
from pathlib import Path
from src.data.loader import WM811KLoader
from src.data.preprocessing import WaferPreprocessor

print('Loading WM-811K dataset...')
t0 = time.time()
loader = WM811KLoader()
records = loader.load(Path('data/raw/LSWMD.pkl'))
summary = loader.get_summary()
print(f'Loaded in {time.time()-t0:.0f}s: {summary["total_count"]:,} records, {summary["labeled_count"]:,} labeled')

print('\nPreprocessing all labeled records (target_size=96)...')
t0 = time.time()
preprocessor = WaferPreprocessor(target_size=96)  # 96x96 for better quality
splits = preprocessor.create_splits(records, seed=42)
print(f'Done in {time.time()-t0:.0f}s')
print(f'Train: {splits["train_data"].shape}')
print(f'Val: {splits["val_data"].shape}')
print(f'Test: {splits["test_data"].shape}')
print(f'Classes: {splits["label_to_index"]}')

# Save
out_dir = Path('data/processed')
preprocessor.save_splits(out_dir, splits)

In [ ]:
# Step 3: Fine-tune ResNet50 (50 epochs)
import torch
from torch.utils.data import DataLoader, TensorDataset
from src.models import get_backbone
from src.training.trainer import Trainer, TrainingConfig

n_classes = int(splits['train_labels'].max().item()) + 1
batch_size = 64  # T4 can handle this easily

train_loader = DataLoader(
    TensorDataset(splits['train_data'], splits['train_labels']),
    batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    TensorDataset(splits['val_data'], splits['val_labels']),
    batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
)

model = get_backbone('resnet50', pretrained=True)
config = TrainingConfig(
    mode='finetune',
    num_epochs=50,
    learning_rate=1e-4,
    weight_decay=1e-5,
    warmup_epochs=5,
    patience=15,
    gradient_clip_max_norm=1.0,
    num_classes=n_classes,
    enable_class_weights=True,
    checkpoint_dir='outputs/checkpoints/resnet50_finetune',
    device='cuda',
    log_every_n_steps=100,
)

trainer = Trainer(config=config, model=model, train_loader=train_loader, val_loader=val_loader)

print(f'Training ResNet50 fine-tune ({n_classes} classes, {len(train_loader.dataset)} samples)...')
t0 = time.time()
result_ft = trainer.train()
print(f'\nDone in {time.time()-t0:.0f}s')
print(f'Best epoch: {result_ft.best_epoch}, Best val loss: {result_ft.best_metric_value:.4f}')

In [ ]:
# Step 4: Metric Learning — Triplet Loss (50 epochs)
from src.data.sampler import BalancedBatchSampler

model_metric = get_backbone('resnet50', pretrained=True)

# Balanced batch sampler: 9 classes x 8 samples = batch of 72
train_labels_list = splits['train_labels'].tolist()
sampler = BalancedBatchSampler(train_labels_list, p_classes=min(9, n_classes), k_samples=8)

metric_train_loader = DataLoader(
    TensorDataset(splits['train_data'], splits['train_labels']),
    batch_sampler=sampler, num_workers=2, pin_memory=True
)

config_metric = TrainingConfig(
    mode='metric',
    loss_name='triplet',
    num_epochs=50,
    learning_rate=1e-4,
    weight_decay=1e-5,
    warmup_epochs=5,
    patience=15,
    margin=0.3,
    gradient_clip_max_norm=1.0,
    projection_hidden=512,
    projection_output=128,
    checkpoint_dir='outputs/checkpoints/resnet50_triplet',
    device='cuda',
    log_every_n_steps=50,
)

trainer_metric = Trainer(
    config=config_metric, model=model_metric,
    train_loader=metric_train_loader, val_loader=val_loader
)

print('Training ResNet50 Metric Learning (Triplet)...')
t0 = time.time()
result_metric = trainer_metric.train()
print(f'\nDone in {time.time()-t0:.0f}s')
print(f'Best epoch: {result_metric.best_epoch}')

In [ ]:
# Step 5: Extract embeddings for all models
import numpy as np
from src.models.embedding_extractor import EmbeddingExtractor

test_loader = DataLoader(
    TensorDataset(splits['test_data'], splits['test_labels']),
    batch_size=128, shuffle=False, num_workers=2, pin_memory=True
)

# Pretrained embeddings
print('Extracting pretrained embeddings...')
model_pt = get_backbone('resnet50', pretrained=True)
ext = EmbeddingExtractor(backbone=model_pt, batch_size=128, device='cuda')
ext.extract(test_loader, Path('outputs/embeddings/resnet50_pretrained_test.npy'))

# Fine-tuned embeddings
print('Extracting fine-tuned embeddings...')
ext_ft = EmbeddingExtractor(backbone=model, batch_size=128, device='cuda')
ext_ft.extract(test_loader, Path('outputs/embeddings/resnet50_finetune_test.npy'))

# Metric learning embeddings
print('Extracting metric learning embeddings...')
ext_ml = EmbeddingExtractor(backbone=model_metric, batch_size=128, device='cuda')
ext_ml.extract(test_loader, Path('outputs/embeddings/resnet50_triplet_test.npy'))

print('All embeddings extracted!')

In [ ]:
# Step 6: Full evaluation and comparison
from src.evaluation.metrics import MetricsComputer
from src.evaluation.benchmark import BenchmarkRunner

labels_np = splits['test_labels'].numpy()
runner = BenchmarkRunner(results_dir=Path('outputs/metrics'))

for name, path in [
    ('resnet50/pretrained', 'outputs/embeddings/resnet50_pretrained_test.npy'),
    ('resnet50/finetune', 'outputs/embeddings/resnet50_finetune_test.npy'),
    ('resnet50/triplet', 'outputs/embeddings/resnet50_triplet_test.npy'),
]:
    emb = np.load(path)
    mc = MetricsComputer(emb, labels_np)
    metrics = mc.compute_all(bootstrap_ci=True, n_bootstrap=500)
    model_name, mode = name.split('/')
    runner.add_result(model_name, mode, metrics)
    print(f'{name}: KNN@5={metrics.knn_accuracy[5]*100:.1f}%, MAP={metrics.mean_average_precision*100:.1f}%, NMI={metrics.nmi:.3f}, Sil={metrics.silhouette_score:.3f}')

# Generate outputs
runner.generate_csv(Path('outputs/metrics/benchmark.csv'))
runner.generate_bar_chart(Path('outputs/metrics/benchmark.png'))
runner.generate_latex(Path('outputs/metrics/benchmark.tex'))

print('\nBest per metric:')
for metric, result in runner.get_best_per_metric().items():
    if metric in ['knn_accuracy_k5', 'mean_average_precision', 'nmi', 'silhouette_score']:
        print(f'  {metric}: {result.model_name}/{result.training_mode}')

In [ ]:
# Step 7: Visualize results
from src.visualization.umap_viz import UMAPVisualizer
import plotly.io as pio

# Best model UMAP
best_emb = np.load('outputs/embeddings/resnet50_triplet_test.npy')
viz = UMAPVisualizer(n_components=2, n_neighbors=15)
proj = viz.fit_transform(best_emb, labels_np)

class_names = list(splits['label_to_index'].keys())
fig = viz.create_figure(proj, labels_np, class_names=class_names, title='ResNet50 Triplet — UMAP')
fig.show()

# Show bar chart
from IPython.display import Image
Image('outputs/metrics/benchmark.png')

In [ ]:
# Step 8: Download results back to your PC
!zip -r results.zip outputs/
from google.colab import files
files.download('results.zip')